In [ ]:
# REVISED CELL 1
import subprocess, sys

# Install compatible versions
packages = [
    "numpy<2.0", 
    "nemo_toolkit[asr]", 
    "soundfile", 
    "huggingface_hub"
]

# Install libraries
subprocess.run([sys.executable, "-m", "pip", "install", "-U"] + packages, check=True)

# Install system dependencies
subprocess.run(["apt-get", "install", "-y", "-q", "libsndfile1", "ffmpeg"], check=True)

print("✅ Installation complete. YOU MUST RESTART THE SESSION NOW.")

# I said RESTART ,,,,,,

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_Token")

In [ ]:
HF_TOKEN        =   secret_value_0    # your HF token

MODEL_REPO_ID   = "teamvillagers/titu-stt-bn-conformer-large-finetuned"
DATASET_REPO_ID = "Sanjidh090/Lipi-Ghor-bn-882-SSTT"   # source audio
OUTPUT_REPO_ID  = "hasans090/conformer_inference_lp"     # your private output repo

DATASET_AUDIO_FOLDER = "data"          # folder inside the dataset repo
OUTPUT_CSV           = "/kaggle/working/conformer_lipighor.csv"
LOCAL_AUDIO_CACHE    = "/kaggle/working/audio_cache"   # one file at a time here
CHUNK_SEC   = 20
OVERLAP_SEC = 2
BATCH_SIZE  = 16   # per GPU — each T4 handles 16 chunks at a time

import os
os.makedirs(LOCAL_AUDIO_CACHE, exist_ok=True)
print("✅ Config ready")

In [ ]:
from huggingface_hub import hf_hub_download
import nemo.collections.asr as nemo_asr
import torch

print(f"Downloading model: {MODEL_REPO_ID}")
nemo_path = hf_hub_download(
    repo_id  = MODEL_REPO_ID,
    filename = "titu_bn_finetuned_lighten.nemo",
    token    = HF_TOKEN,
)

# Load one model per GPU
print("Loading model on GPU 0...")
model_0 = nemo_asr.models.ASRModel.restore_from(nemo_path)
model_0 = model_0.to("cuda:0")
model_0.eval()

print("Loading model on GPU 1...")
model_1 = nemo_asr.models.ASRModel.restore_from(nemo_path)
model_1 = model_1.to("cuda:1")
model_1.eval()

models = [model_0, model_1]
print(f"✅ Both models loaded")
print(f"   GPU 0: {torch.cuda.get_device_name(0)}")
print(f"   GPU 1: {torch.cuda.get_device_name(1)}")

In [ ]:
from omegaconf import OmegaConf

for idx, m in enumerate(models):
    m.change_decoding_strategy(OmegaConf.create({"strategy": "greedy_batch"}))
    print(f"✅ GPU {idx}: greedy_batch")

In [ ]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
api.create_repo(
    repo_id   = OUTPUT_REPO_ID,
    repo_type = "dataset",
    private   = True,
    exist_ok  = True,
)
print(f"✅ Repo ready: https://huggingface.co/datasets/{OUTPUT_REPO_ID}")

In [ ]:
import os, torch, tempfile, subprocess
from pathlib import Path
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from huggingface_hub import hf_hub_download, list_repo_files, upload_file

def get_duration(path):
    result = subprocess.run([
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        str(path)
    ], capture_output=True, text=True)
    return float(result.stdout.strip())

def extract_chunk_ffmpeg(audio_path, start_sec, duration_sec, target_sr=16000):
    tmp = tempfile.mktemp(suffix=".wav")
    subprocess.run([
        "ffmpeg", "-y",
        "-ss", str(start_sec),
        "-t",  str(duration_sec),
        "-i",  str(audio_path),
        "-ar", str(target_sr),
        "-ac", "1", tmp
    ], capture_output=True, check=True)
    return tmp

def extract_text(hyp_item):
    if isinstance(hyp_item, str):       return hyp_item
    elif hasattr(hyp_item, "text"):     return hyp_item.text
    elif hasattr(hyp_item, "__iter__"): return hyp_item[0]
    return str(hyp_item)

def dedup_overlap(t1, t2, max_w=8):
    w1, w2 = t1.split(), t2.split()
    for n in range(min(max_w, len(w1), len(w2)), 0, -1):
        if w1[-n:] == w2[:n]:
            return n
    return 0

def list_audio_files_on_hf(dataset_repo, folder, token):
    AUDIO_EXTS = {".wav", ".mp3", ".flac", ".m4a", ".ogg"}
    all_files  = list_repo_files(dataset_repo, repo_type="dataset", token=token)
    return sorted([
        f for f in all_files
        if f.startswith(folder + "/") and Path(f).suffix.lower() in AUDIO_EXTS
    ])

def download_single_audio(dataset_repo, hf_path, local_dir, token):
    return hf_hub_download(
        repo_id   = dataset_repo,
        filename  = hf_path,
        repo_type = "dataset",
        token     = token,
        local_dir = local_dir,
    )

def push_csv_to_hf(local_csv, output_repo, token):
    upload_file(
        path_or_fileobj = local_csv,
        path_in_repo    = Path(local_csv).name,
        repo_id         = output_repo,
        repo_type       = "dataset",
        token           = token,
    )

def transcribe_on_gpu(model, tmp_files, batch_size):
    """Transcribe a list of wav files on whichever GPU model lives on."""
    with torch.no_grad():
        hyps = model.transcribe(tmp_files, batch_size=batch_size)
    return [extract_text(h) for h in hyps]

def transcribe_dual_gpu(tmp_files, models, batch_size):
    """Split chunks across 2 GPUs and run in parallel threads."""
    mid   = len(tmp_files) // 2
    half0 = tmp_files[:mid]
    half1 = tmp_files[mid:]

    with ThreadPoolExecutor(max_workers=2) as ex:
        f0 = ex.submit(transcribe_on_gpu, models[0], half0, batch_size)
        f1 = ex.submit(transcribe_on_gpu, models[1], half1, batch_size)
        res0 = f0.result()
        res1 = f1.result()

    return res0 + res1

print("✅ Utilities ready")

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path
import pandas as pd, os, torch

# ── List all audio files on HF ────────────────────────────────────────────────
hf_audio_paths = list_audio_files_on_hf(DATASET_REPO_ID, DATASET_AUDIO_FOLDER, HF_TOKEN)
TOTAL = len(hf_audio_paths)
print(f"Found {TOTAL} audio files in repo\n")

# ── Resume: HF CSV → local CSV → fresh start ─────────────────────────────────
def load_existing_csv_from_hf(output_repo, csv_filename, token):
    try:
        path = hf_hub_download(
            repo_id        = output_repo,
            filename       = csv_filename,
            repo_type      = "dataset",
            token          = token,
            force_download = True,
        )
        df = pd.read_csv(path)
        print(f"✅ Loaded from HF — {len(df)} rows done so far")
        return df
    except Exception as e:
        print(f"ℹ️ No existing CSV on HF (starting fresh): {e}")
        return None

hf_df = load_existing_csv_from_hf(OUTPUT_REPO_ID, Path(OUTPUT_CSV).name, HF_TOKEN)

if hf_df is not None:
    done_ids = set(hf_df["id"].astype(str).tolist())
    results  = hf_df.to_dict("records")
    hf_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
    print(f"Resuming from HF — {len(done_ids)} done")
elif Path(OUTPUT_CSV).exists():
    done_df  = pd.read_csv(OUTPUT_CSV)
    done_ids = set(done_df["id"].astype(str).tolist())
    results  = done_df.to_dict("records")
    print(f"Resuming from local — {len(done_ids)} done")
else:
    done_ids = set()
    results  = []
    print("Starting fresh")

print(f"\n📊 {len(done_ids)} done / {TOTAL} total — {TOTAL - len(done_ids)} remaining\n")

# ── Inference loop ────────────────────────────────────────────────────────────
for i, hf_path in enumerate(hf_audio_paths):
    file_id = Path(hf_path).stem

    if file_id in done_ids:
        continue

    os.system("clear")
    print(f"{'─'*60}")
    print(f"  [{len(results)+1}/{TOTAL}]  {file_id}")
    print(f"{'─'*60}\n")

    try:
        print("⬇  Downloading...")
        audio_path = download_single_audio(DATASET_REPO_ID, hf_path, LOCAL_AUDIO_CACHE, HF_TOKEN)
        print(f"   Saved to: {audio_path}")

        total_sec = get_duration(audio_path)
        print(f"⏱  Duration : {total_sec:.0f}s ({total_sec/60:.1f} min)")

        step = CHUNK_SEC - OVERLAP_SEC
        starts, pos = [], 0.0
        while pos < total_sec:
            starts.append(pos)
            if pos + CHUNK_SEC >= total_sec: break
            pos += step
        print(f"🔪  Chunks   : {len(starts)}  →  split {len(starts)//2} | {len(starts) - len(starts)//2} across 2 GPUs\n")

        # ── Extract all chunks ────────────────────────────────────────────────
        tmp_files = []
        for start in starts:
            dur = min(CHUNK_SEC, total_sec - start)
            tmp_files.append(extract_chunk_ffmpeg(audio_path, start, dur))

        # ── Dual GPU batch transcribe ─────────────────────────────────────────
        chunk_texts = transcribe_dual_gpu(tmp_files, models, BATCH_SIZE)

        for tmp in tmp_files:
            os.remove(tmp)

        for j, text in enumerate(chunk_texts):
            print(f"  chunk {j+1:>3}/{len(starts)}: {text[:70]}")

        # ── Stitch ────────────────────────────────────────────────────────────
        words = chunk_texts[0].split() if chunk_texts else []
        for k in range(1, len(chunk_texts)):
            skip = dedup_overlap(chunk_texts[k-1], chunk_texts[k]) if OVERLAP_SEC > 0 else 0
            words.extend(chunk_texts[k].split()[skip:])
        transcript = " ".join(words).strip()

        os.remove(audio_path)
        print(f"\n✅  {len(transcript.split())} words total")

    except Exception as ex:
        transcript = ""
        print(f"\n❌  ERROR: {ex}")

    results.append({"id": file_id, "transcript": transcript})
    pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

    files_done = len(results)
    is_last    = (i == TOTAL - 1)
    if files_done % 10 == 0 or is_last:
        try:
            push_csv_to_hf(OUTPUT_CSV, OUTPUT_REPO_ID, HF_TOKEN)
            print(f"📤  CSV pushed to HF ({files_done} rows)")
        except Exception as e:
            print(f"⚠️  Push failed: {e}")

print(f"\n✅ All done — {len(results)} rows")
print(pd.read_csv(OUTPUT_CSV)[["id", "transcript"]].head(10).to_string(index=False))